# Behind the pipeline (PyTorch)

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [1]:
!pip install datasets evaluate transformers[sentencepiece]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00


In [2]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
classifier(
    [
        "I've been waiting for a HuggingFace course my whole life.",
        "I hate this so much!",
    ]
)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9598046541213989},
 {'label': 'NEGATIVE', 'score': 0.9994558691978455}]

In [3]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [4]:
raw_inputs = [
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!",
]
inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt")
print(inputs)

{'input_ids': tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,
          2607,  2026,  2878,  2166,  1012,   102],
        [  101,  1045,  5223,  2023,  2061,  2172,   999,   102,     0,     0,
             0,     0,     0,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]])}


In [5]:
from transformers import AutoModel

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModel.from_pretrained(checkpoint)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key                   | Status     |  | 
----------------------+------------+--+-
classifier.bias       | UNEXPECTED |  | 
pre_classifier.bias   | UNEXPECTED |  | 
pre_classifier.weight | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
outputs = model(**inputs)
print(outputs.last_hidden_state.shape)

torch.Size([2, 16, 768])


In [7]:
from transformers import AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
outputs = model(**inputs)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [8]:
print(outputs.logits.shape)

torch.Size([2, 2])


In [9]:
print(outputs.logits)

tensor([[-1.5607,  1.6123],
        [ 4.1692, -3.3464]], grad_fn=<AddmmBackward0>)


In [10]:
import torch

predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
print(predictions)

tensor([[4.0195e-02, 9.5980e-01],
        [9.9946e-01, 5.4418e-04]], grad_fn=<SoftmaxBackward0>)


In [11]:
model.config.id2label

{0: 'NEGATIVE', 1: 'POSITIVE'}

In [12]:
# Try your own custom sentences for sentiment analysis
my_inputs = [
    "I love learning about AI and machine learning!",
    "This homework is really boring and difficult.",
    "The weather today is just okay, nothing special.",
]
classifier = pipeline("sentiment-analysis")
results = classifier(my_inputs)
for sentence, result in zip(my_inputs, results):
    print(f"Text: {sentence}")
    print(f"Label: {result['label']} | Score: {round(result['score'], 4)}")
    print("---")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Text: I love learning about AI and machine learning!
Label: POSITIVE | Score: 0.9998
---
Text: This homework is really boring and difficult.
Label: NEGATIVE | Score: 0.9994
---
Text: The weather today is just okay, nothing special.
Label: NEGATIVE | Score: 0.6382
---


In [13]:
# See how the tokenizer breaks sentences into tokens
tokenizer_viz = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

test_sentence = "I've been waiting for a HuggingFace course my whole life."
tokens = tokenizer_viz.tokenize(test_sentence)
print("Tokens:", tokens)
print("Total tokens:", len(tokens))

Tokens: ['i', "'", 've', 'been', 'waiting', 'for', 'a', 'hugging', '##face', 'course', 'my', 'whole', 'life', '.']
Total tokens: 14


In [14]:
# Convert tokens → IDs → back to words
token_ids = tokenizer_viz.convert_tokens_to_ids(tokens)
print("Token IDs:", token_ids)

# Decode back to original text
decoded = tokenizer_viz.decode(token_ids)
print("Decoded back:", decoded)

Token IDs: [1045, 1005, 2310, 2042, 3403, 2005, 1037, 17662, 12172, 2607, 2026, 2878, 2166, 1012]
Decoded back: i ' ve been waiting for a huggingface course my whole life.


In [15]:
# Compare raw logits vs softmax probabilities side by side
import torch

sentences = ["I love this!", "I hate this!"]
inputs = tokenizer_viz(sentences, padding=True, truncation=True, return_tensors="pt")

model_sc = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased-finetuned-sst-2-english"
)
outputs = model_sc(**inputs)

logits = outputs.logits
probs = torch.nn.functional.softmax(logits, dim=-1)

for i, sentence in enumerate(sentences):
    print(f"Sentence: {sentence}")
    print(f"  Raw Logits  → NEGATIVE: {logits[i][0].item():.4f} | POSITIVE: {logits[i][1].item():.4f}")
    print(f"  Probability → NEGATIVE: {probs[i][0].item():.4f} | POSITIVE: {probs[i][1].item():.4f}")
    print("---")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Sentence: I love this!
  Raw Logits  → NEGATIVE: -4.3222 | POSITIVE: 4.6768
  Probability → NEGATIVE: 0.0001 | POSITIVE: 0.9999
---
Sentence: I hate this!
  Raw Logits  → NEGATIVE: 4.3139 | POSITIVE: -3.4527
  Probability → NEGATIVE: 0.9996 | POSITIVE: 0.0004
---


In [16]:
# Test how the model handles tricky/ambiguous sentences
tricky_sentences = [
    "Not bad at all!",           # Double negative - should be positive
    "I'm not happy about this.", # Negation - should be negative
    "Could be worse.",           # Ambiguous
    "Absolutely terrible... NOT! It's amazing!", # Sarcasm
]

results = classifier(tricky_sentences)
for sentence, result in zip(tricky_sentences, results):
    print(f"Text : {sentence}")
    print(f"Label: {result['label']} | Score: {round(result['score'], 4)}")
    print("---")

Text : Not bad at all!
Label: POSITIVE | Score: 0.9996
---
Text : I'm not happy about this.
Label: NEGATIVE | Score: 0.9998
---
Text : Could be worse.
Label: NEGATIVE | Score: 0.9996
---
Text : Absolutely terrible... NOT! It's amazing!
Label: NEGATIVE | Score: 0.9546
---
